In [1]:
import tensorflow as tf
import numpy as np

Definindo os inputs da rede neural. Essa rede neural será capaz de aprender a função "AND" da lógica proposicional.

In [2]:
X = np.array([[0.0, 0.0],
              [0.0, 1.0],
              [1.0, 0.0],
              [1.0, 1.0]])

X

array([[0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 1.]])

Definindo o "dataset" (Respostas da rede neural para cada input)

In [3]:
#False, False, False, True

Y = np.array([[0.0] ,[0.0] ,[0.0] , [1.0]])

Y

array([[0.],
       [0.],
       [0.],
       [1.]])

Definindo o vetor dos pesos da rede neural, uma matriz 2x1 preenchida com zeros (Definida como uma variavel do tf)

OBS: W precisou ser definido como float64 pois é o mesmo tipo de dados de X que foi criado como float64 por padrão.

In [4]:
W = tf.Variable(tf.zeros([2, 1], dtype = tf.float64))

W

<tf.Variable 'Variable:0' shape=(2, 1) dtype=float64, numpy=
array([[0.],
       [0.]])>

Definindo a soma da rede neural, fara um produto interno com X e W:

In [5]:
soma = tf.matmul(X, W)

soma

<tf.Tensor: shape=(4, 1), dtype=float64, numpy=
array([[0.],
       [0.],
       [0.],
       [0.]])>

Definindo uma função stepFunction usando tensorflow.

Será implementada usando a função greater_equal, que retorna True se o 1o argumento for maior que o segundo, e falso caso contrário.

Em seguida, esse resultado booleano será castado para float64

In [6]:
def step(x):
    return tf.cast(tf.greater_equal(x, 1), tf.float64)

step(0)

<tf.Tensor: shape=(), dtype=float64, numpy=0.0>

Agora será aplicada função de ativação na saída da soma do neuronio, para que o resultado possa ser passado para frente

Pode-se observar que o tensorflow aplica automaticamente a funcao step definida acima, para cada elemento do tensor/array, sem precisar fazer isso explicitamente. Isso se chama broadcasting.

Em outras palavras, de alguma maneira ele sabe que se receber um x de qualquer tamanho na função step, ele aplicara ela elemento a elemento.

In [7]:
ativacao = step(soma)

ativacao

<tf.Tensor: shape=(4, 1), dtype=float64, numpy=
array([[0.],
       [0.],
       [0.],
       [0.]])>

Definindo o erro de cada input, será definido como:

ValorEsperado - ValorOutputado pelo neuronio

In [8]:
erro = tf.subtract(Y, ativacao)

erro

<tf.Tensor: shape=(4, 1), dtype=float64, numpy=
array([[0.],
       [0.],
       [0.],
       [1.]])>

Calculando o delta que será usado para inferir a direção da descida do gradiente para atualizar os pesos. Será calculado fazendo um produto interno entre a transposta de X, e o erro (Ver por que)

In [9]:
X_transposed = tf.transpose(X)

print(X)
print(X_transposed)

#delta = tf.matmul(X, erro, transpose_a = True) --> Para não precisar fazer tf.transpose(X)
delta = tf.matmul(X_transposed, erro)
delta

[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
tf.Tensor(
[[0. 0. 1. 1.]
 [0. 1. 0. 1.]], shape=(2, 4), dtype=float64)


<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[1.],
       [1.]])>

Agora o delta sera multiplicado pela learning rate, de alguma maneira implementando a formula:

Somatório(Entrada * delta) * LearningRate.

(Ver como)

In [10]:
learning_rate = 0.1
treinamento = tf.multiply(delta, learning_rate)

treinamento

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[0.1],
       [0.1]])>

Agora o peso será introduzido na formula, fazendo ela:

pesoNovo = pesoAntigo*momento * Somatório(Entrada * delta) * LearningRate

In [11]:
momento = 1

treinamento = tf.add(W*momento, treinamento)

treinamento

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[0.1],
       [0.1]])>

In [12]:
#Para atualizar os pesos

W.assign(treinamento)

W

<tf.Variable 'Variable:0' shape=(2, 1) dtype=float64, numpy=
array([[0.1],
       [0.1]])>

Com a formula para atualização dos pesos pronta, pode-se começar o treinamento do perceptron de uma camada (Definirei tudo novamente dentro do for pois o curso comprado é em TF1, mas estou usando o TF2 (Sem session)):

In [ ]:
epoch = 0
for i in range(15):
    epoch += 1

    soma = tf.matmul(X, W)
    ativacao = step(soma)

    #Erro de cada entrada
    errors = tf.subtract(Y, ativacao)

    delta = tf.matmul(X, errors, transpose_a = True)
    treinamento = tf.add(W*momento, tf.multiply(delta, learning_rate))
    W.assign(treinamento) 

    #Erro total
    errors_sum = tf.reduce_sum(errors).numpy()
    print('Época:', epoch, ' Erro: ', errors_sum)

    if errors_sum == 0.0:
        break
    

Época: 1  Erro:  1.0
Época: 2  Erro:  1.0
Época: 3  Erro:  1.0
Época: 4  Erro:  1.0
Época: 5  Erro:  0.0


Agora testarei o perceptron treinado para verificar os valores retornados pelo neuronio dentro da função AND

In [14]:
teste_sum = tf.matmul(X, W)
teste_ativacao = step(teste_sum)

teste_ativacao

<tf.Tensor: shape=(4, 1), dtype=float64, numpy=
array([[0.],
       [0.],
       [0.],
       [1.]])>

Também pode-se repetir o processo para um perceptron aprender a função OR

In [16]:
def treino(X, Y):
    
    epoch = 0
    W = tf.Variable(tf.zeros([2, 1], dtype = tf.float64))

    for i in range(15):
        epoch += 1

        soma = tf.matmul(X, W)
        ativacao = step(soma)

        #Erro de cada entrada
        errors = tf.subtract(Y, ativacao)

        delta = tf.matmul(X, errors, transpose_a = True)
        treinamento = tf.add(W*momento, tf.multiply(delta, learning_rate))
        W.assign(treinamento) 

        #Erro total
        errors_sum = tf.reduce_sum(errors).numpy()
        print('Época:', epoch, ' Erro: ', errors_sum)

        if errors_sum == 0.0:
            break

    return W

def teste(X, W):
    teste_sum = tf.matmul(X, W)
    teste_ativacao = step(teste_sum)

    print(teste_ativacao)

    
X_OR = np.array([[0.0, 0.0],
              [0.0, 1.0],
              [1.0, 0.0],
              [1.0, 1.0]])

Y_OR = np.array([[0.0], [1.0], [1.0], [1.0]])


W_OR = treino(X_OR, Y_OR)
teste(X_OR, W_OR)


Época: 1  Erro:  3.0
Época: 2  Erro:  3.0
Época: 3  Erro:  3.0
Época: 4  Erro:  2.0
Época: 5  Erro:  2.0
Época: 6  Erro:  2.0
Época: 7  Erro:  2.0
Época: 8  Erro:  0.0
tf.Tensor(
[[0.]
 [1.]
 [1.]
 [1.]], shape=(4, 1), dtype=float64)
